# 🎲 RandomizedSearchCV — Efficient Hyperparameter Tuning

> **Folder:** `07_Hyperparameter_Tuning`  
> **Notebook:** `randomsearchcv.ipynb`  
> **Author:** Hamna Munir

---

## 🎯 Objectives

By the end of this notebook, you will:

- Understand **why Random Search outperforms Grid Search** for large spaces
- Use **scipy distributions** (loguniform, randint, uniform) for continuous sampling
- Run RandomizedSearchCV on **classification and regression** problems
- Analyze **convergence** — how quickly the best score improves with iterations
- Perform **n_iter sensitivity analysis** — how many trials are enough?
- Tune **XGBoost, LightGBM, and CatBoost** with Random Search
- Build a **post-search analysis** with parameter importance plots
- Use Random Search correctly inside a **Pipeline**

---

## 📚 Techniques Covered

| # | Technique | Key Insight |
|---|-----------|-------------|
| 1 | Dataset Setup | Classification + Regression |
| 2 | Distribution Types | loguniform, randint, uniform, choice |
| 3 | RandomizedSearchCV — RF | Core usage and result analysis |
| 4 | Convergence Analysis | Best score vs n_iter |
| 5 | n_iter Sensitivity | How many trials are enough? |
| 6 | Boosting Models Tuning | XGBoost, LightGBM, CatBoost |
| 7 | Parameter Importance | Which params matter most? |
| 8 | Regression Tuning | RandomizedSearch for regression |
| 9 | Pipeline + RandomSearch | Correct leakage-free usage |
| 10 | Multi-Model Leaderboard | Best tuned model across all algorithms |
| 11 | Summary & Golden Rules | Key takeaways |


---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    RandomizedSearchCV, GridSearchCV,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    r2_score, mean_squared_error,
)
from scipy.stats import randint, uniform, loguniform

# Optional boosting libraries
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed — skipping XGB cells")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("LightGBM not installed — skipping LGB cells")

try:
    from catboost import CatBoostClassifier
    CAT_AVAILABLE = True
except ImportError:
    CAT_AVAILABLE = False
    print("CatBoost not installed — skipping CatBoost cells")

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)

COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'warning'  : '#F18F01',
    'purple'   : '#7B2D8B',
    'palette'  : ['#2E86AB','#E84855','#3BB273','#F18F01','#7B2D8B','#F4D35E'],
}
print('✅ Libraries loaded successfully!')

---
## 1️⃣ Dataset Setup


In [ ]:
np.random.seed(42)

# ── Binary classification ─────────────────────────────────────────────────
X_clf, y_clf = make_classification(
    n_samples=1000, n_features=20, n_informative=10,
    n_redundant=5, n_classes=2, weights=[0.5, 0.5],
    random_state=42
)
feat_names = [f'F{i+1:02d}' for i in range(20)]
X_clf_df   = pd.DataFrame(X_clf, columns=feat_names)
y_clf_s    = pd.Series(y_clf, name='Target')

# ── Regression ────────────────────────────────────────────────────────────
X_reg, y_reg = make_regression(
    n_samples=700, n_features=15, n_informative=8,
    noise=20, random_state=42
)
X_reg_df = pd.DataFrame(X_reg, columns=[f'R{i+1:02d}' for i in range(15)])
y_reg_s  = pd.Series(y_reg, name='Target')

# ── Scale + split ─────────────────────────────────────────────────────────
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf_df, y_clf_s, test_size=0.2, stratify=y_clf_s, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg_df, y_reg_s, test_size=0.2, random_state=42)

sc_clf = StandardScaler()
sc_reg = StandardScaler()
Xc_tr_sc = sc_clf.fit_transform(Xc_tr); Xc_te_sc = sc_clf.transform(Xc_te)
Xr_tr_sc = sc_reg.fit_transform(Xr_tr); Xr_te_sc = sc_reg.transform(Xr_te)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kf  = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Classification : {X_clf_df.shape} | classes={dict(y_clf_s.value_counts().sort_index())}')
print(f'  Train={Xc_tr.shape}  Test={Xc_te.shape}')
print(f'Regression     : {X_reg_df.shape}')
print(f'  Train={Xr_tr.shape}  Test={Xr_te.shape}')

---
## 2️⃣ Sampling Distributions — The Key Advantage of Random Search

> Unlike Grid Search (discrete lists only), Random Search can sample from  
> **continuous distributions** — giving finer coverage of the search space.
>
> | Distribution | Use For | Example |
> |-------------|---------|---------|
> | `loguniform(a, b)` | Rates, regularization (log scale) | learning_rate, C, alpha |
> | `uniform(a, width)` | Fractions, probabilities | subsample ∈ [0.5, 1.0] |
> | `randint(low, high)` | Integers | n_estimators, max_depth |
> | List / Categorical | Discrete choices | kernel, criterion |


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
n_samples = 2000

dists = [
    ('loguniform(0.001, 1.0)
learning_rate / C / alpha',
     loguniform(0.001, 1.0).rvs(n_samples, random_state=42),
     True, COLORS['primary']),
    ('uniform(0.5, 0.5)
subsample ∈ [0.5, 1.0]',
     uniform(0.5, 0.5).rvs(n_samples, random_state=42),
     False, COLORS['accent']),
    ('randint(50, 500)
n_estimators',
     randint(50, 500).rvs(n_samples, random_state=42),
     False, COLORS['secondary']),
    ('loguniform(0.0001, 0.1)
gamma for SVM/RBF',
     loguniform(0.0001, 0.1).rvs(n_samples, random_state=42),
     True, COLORS['warning']),
    ('uniform(0.0, 1.0)
L1 ratio / dropout',
     uniform(0.0, 1.0).rvs(n_samples, random_state=42),
     False, COLORS['purple']),
    ('randint(2, 15)
max_depth',
     randint(2, 15).rvs(n_samples, random_state=42),
     False, COLORS['palette'][5]),
]

for ax, (title, samples, log_scale, color) in zip(axes.flatten(), dists):
    ax.hist(samples, bins=40, color=color, alpha=0.80, edgecolor='white')
    if log_scale:
        ax.set_xscale('log')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Frequency', fontsize=9)
    ax.axvline(np.median(samples), color='black', linestyle='--',
               linewidth=1.5, label=f'Median={np.median(samples):.4f}')
    ax.legend(fontsize=8)

plt.suptitle('Sampling Distributions for RandomizedSearchCV',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight: loguniform gives EQUAL probability to each order of magnitude')
print('  → [0.001, 0.01] same probability as [0.01, 0.1] as [0.1, 1.0]')
print('  → Ideal for learning_rate, C, alpha, gamma')

---
## 3️⃣ RandomizedSearchCV — Core Usage (Random Forest)

> `n_iter` controls how many random configurations to evaluate.  
> Each configuration is evaluated with `cv` folds — total fits = `n_iter × cv`.


In [ ]:
param_dist_rf = {
    'n_estimators'     : randint(50, 500),
    'max_depth'        : randint(2, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf' : randint(1, 15),
    'max_features'     : uniform(0.1, 0.9),
    'bootstrap'        : [True, False],
}

n_iter = 80
rs_rf  = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist_rf,
    n_iter=n_iter,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=0,
    return_train_score=True,
)
rs_rf.fit(Xc_tr_sc, yc_tr)

test_auc = roc_auc_score(yc_te, rs_rf.predict_proba(Xc_te_sc)[:,1])
test_acc = accuracy_score(yc_te, rs_rf.predict(Xc_te_sc))
test_f1  = f1_score(yc_te, rs_rf.predict(Xc_te_sc))

print('RandomizedSearchCV — Random Forest')
print(f'  n_iter        : {n_iter} (total fits = {n_iter*5})')
print(f'  Best params   : {rs_rf.best_params_}')
print(f'  Best CV AUC   : {rs_rf.best_score_:.4f}')
print(f'  Test AUC      : {test_auc:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}')
print(f'  Test F1       : {test_f1:.4f}')

# Results DataFrame
cv_df = pd.DataFrame(rs_rf.cv_results_)
print(f'\nTop 10 Configurations:')
top10 = cv_df.nlargest(10, 'mean_test_score')[[
    'param_n_estimators','param_max_depth','param_min_samples_leaf',
    'param_max_features','mean_test_score','std_test_score','mean_train_score'
]].round(4)
top10.columns = ['n_est','depth','min_leaf','max_feat','Val AUC','Val Std','Train AUC']
print(top10.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Score distribution
axes[0].hist(cv_df['mean_test_score'], bins=20,
             color=COLORS['primary'], alpha=0.80, edgecolor='white')
axes[0].axvline(rs_rf.best_score_, color=COLORS['secondary'],
                linestyle='--', linewidth=2.5,
                label=f'Best={rs_rf.best_score_:.4f}')
axes[0].axvline(cv_df['mean_test_score'].mean(), color=COLORS['warning'],
                linestyle=':', linewidth=2,
                label=f'Mean={cv_df["mean_test_score"].mean():.4f}')
axes[0].set_xlabel('CV ROC-AUC', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Score Distribution — All Configurations', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# Train vs Val scatter
axes[1].scatter(cv_df['mean_train_score'], cv_df['mean_test_score'],
                c=cv_df['mean_test_score'], cmap='RdYlGn',
                s=50, alpha=0.7, edgecolors='white')
axes[1].plot([0.5,1.0],[0.5,1.0], 'k--', linewidth=1.5, label='No overfit line')
axes[1].set_xlabel('Train AUC', fontsize=11)
axes[1].set_ylabel('Validation AUC', fontsize=11)
axes[1].set_title('Train vs Validation AUC
(above diagonal = overfitting)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('RandomizedSearchCV — Random Forest Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4️⃣ Convergence Analysis — Best Score vs Iteration

> How quickly does RandomizedSearch find a good configuration?  
> The convergence plot shows the **cumulative best score** over iterations.
>
> A flat plateau indicates the search has converged — additional iterations  
> are unlikely to improve the result further.


In [ ]:
all_scores   = cv_df['mean_test_score'].values
best_so_far  = np.maximum.accumulate(all_scores)
improvement  = np.diff(best_so_far, prepend=best_so_far[0])

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Convergence
axes[0].plot(range(1, n_iter+1), all_scores, 'o',
             color=COLORS['primary'], alpha=0.35, markersize=5, label='Each trial')
axes[0].plot(range(1, n_iter+1), best_so_far, '-',
             color=COLORS['secondary'], linewidth=2.5, label='Best so far')
axes[0].axhline(rs_rf.best_score_, color=COLORS['accent'],
                linestyle='--', linewidth=2,
                label=f'Final best={rs_rf.best_score_:.4f}')
first_best_iter = np.argmax(best_so_far >= rs_rf.best_score_) + 1
axes[0].axvline(first_best_iter, color=COLORS['warning'],
                linestyle=':', linewidth=2,
                label=f'Best found at iter={first_best_iter}')
axes[0].set_xlabel('Iteration', fontsize=11)
axes[0].set_ylabel('CV ROC-AUC', fontsize=11)
axes[0].set_title('Convergence — Best Score Over Iterations',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8)

# Marginal improvement
axes[1].bar(range(1, n_iter+1), improvement,
            color=[COLORS['accent'] if v > 0 else COLORS['secondary'] for v in improvement],
            alpha=0.75, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Improvement over previous best', fontsize=11)
axes[1].set_title('Marginal Improvement per Iteration',
                  fontsize=12, fontweight='bold')

# Cumulative % of final best achieved
pct_of_best = best_so_far / rs_rf.best_score_ * 100
axes[2].plot(range(1, n_iter+1), pct_of_best, '-',
             color=COLORS['primary'], linewidth=2.5)
for thresh in [90, 95, 99]:
    idx = np.argmax(pct_of_best >= thresh)
    if pct_of_best[idx] >= thresh:
        axes[2].axhline(thresh, color='gray', linestyle=':', linewidth=1.2)
        axes[2].axvline(idx+1, color='gray', linestyle=':', linewidth=1.2)
        axes[2].text(idx+2, thresh+0.3, f'{thresh}% @ iter {idx+1}',
                     fontsize=8, color='gray')
axes[2].set_xlabel('Iteration', fontsize=11)
axes[2].set_ylabel('% of Final Best Score', fontsize=11)
axes[2].set_title('% of Final Best Achieved vs Iteration',
                  fontsize=12, fontweight='bold')
axes[2].set_ylim([80, 101])

plt.suptitle('RandomizedSearchCV — Convergence Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Best score found at iteration: {first_best_iter} / {n_iter}')
print(f'90% of best achieved by iter: {np.argmax(pct_of_best >= 90)+1}')
print(f'95% of best achieved by iter: {np.argmax(pct_of_best >= 95)+1}')
print(f'99% of best achieved by iter: {np.argmax(pct_of_best >= 99)+1}')

---
## 5️⃣ n_iter Sensitivity — How Many Trials Are Enough?

> Running RandomizedSearch with different `n_iter` values reveals  
> the point of **diminishing returns** — beyond which more trials  
> add little improvement.
>
> Rule of thumb: **n_iter = 50–100** is sufficient for most models.  
> For very large search spaces or expensive models: n_iter = 100–200.


In [ ]:
n_iter_values = [5, 10, 20, 30, 50, 75, 100, 150]
n_repeats     = 5  # repeat each n_iter with different seeds for stability

iter_results = []
for n_it in n_iter_values:
    scores_per_iter = []
    for seed in range(n_repeats):
        rs = RandomizedSearchCV(
            RandomForestClassifier(random_state=42),
            param_dist_rf,
            n_iter=n_it, cv=skf, scoring='roc_auc',
            n_jobs=-1, random_state=seed
        )
        rs.fit(Xc_tr_sc, yc_tr)
        scores_per_iter.append(rs.best_score_)
    iter_results.append({
        'n_iter' : n_it,
        'Mean'   : round(np.mean(scores_per_iter), 4),
        'Std'    : round(np.std(scores_per_iter), 4),
        'Min'    : round(np.min(scores_per_iter), 4),
        'Max'    : round(np.max(scores_per_iter), 4),
    })
    print(f'  n_iter={n_it:3d}: mean={np.mean(scores_per_iter):.4f} '
          f'± {np.std(scores_per_iter):.4f}  '
          f'[{np.min(scores_per_iter):.4f}, {np.max(scores_per_iter):.4f}]')

iter_df = pd.DataFrame(iter_results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(iter_df['n_iter'], iter_df['Mean'], 'o-',
        color=COLORS['primary'], linewidth=2.5, markersize=9, label='Mean best score')
ax.fill_between(iter_df['n_iter'],
                iter_df['Mean'] - iter_df['Std'],
                iter_df['Mean'] + iter_df['Std'],
                alpha=0.20, color=COLORS['primary'], label='±1 std (across seeds)')
ax.fill_between(iter_df['n_iter'],
                iter_df['Min'], iter_df['Max'],
                alpha=0.08, color=COLORS['secondary'], label='Min–Max range')
ax.axvline(50, color=COLORS['warning'], linestyle='--',
           linewidth=2, label='n_iter=50 (recommended minimum)')
for _, row in iter_df.iterrows():
    ax.annotate(f'{row["Mean"]:.4f}',
                (row['n_iter'], row['Mean']),
                textcoords='offset points', xytext=(5, 6), fontsize=8)
ax.set_xlabel('n_iter', fontsize=11)
ax.set_ylabel('Best CV ROC-AUC', fontsize=11)
ax.set_title(f'n_iter Sensitivity ({n_repeats} seeds per value) — Diminishing Returns',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 6️⃣ Tuning Boosting Models — XGBoost, LightGBM, CatBoost

> Boosting models have many hyperparameters — Random Search is the  
> most practical tuning strategy before Bayesian optimization.
>
> Key parameters per model:
> - **XGBoost**: `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `reg_alpha`, `reg_lambda`
> - **LightGBM**: `n_estimators`, `num_leaves`, `learning_rate`, `subsample`, `colsample_bytree`, `reg_alpha`, `min_child_samples`
> - **CatBoost**: `iterations`, `depth`, `learning_rate`, `l2_leaf_reg`, `rsm`


In [ ]:
boosting_results = []

# ── XGBoost ───────────────────────────────────────────────────────────────
if XGB_AVAILABLE:
    xgb_dist = {
        'n_estimators'    : randint(100, 500),
        'max_depth'       : randint(2, 10),
        'learning_rate'   : loguniform(0.005, 0.3),
        'subsample'       : uniform(0.5, 0.5),
        'colsample_bytree': uniform(0.4, 0.6),
        'reg_alpha'       : loguniform(1e-4, 10),
        'reg_lambda'      : loguniform(1e-4, 10),
        'min_child_weight': randint(1, 10),
    }
    rs_xgb = RandomizedSearchCV(
        xgb.XGBClassifier(tree_method='hist', use_label_encoder=False,
                           eval_metric='logloss', random_state=42, verbosity=0),
        xgb_dist, n_iter=40, cv=skf, scoring='roc_auc',
        n_jobs=-1, random_state=42
    )
    rs_xgb.fit(Xc_tr_sc, yc_tr)
    auc_xgb = roc_auc_score(yc_te, rs_xgb.predict_proba(Xc_te_sc)[:,1])
    boosting_results.append({'Model':'XGBoost','CV AUC':round(rs_xgb.best_score_,4),
                              'Test AUC':round(auc_xgb,4),
                              'Best Params':str(rs_xgb.best_params_)})
    print(f'XGBoost  : CV={rs_xgb.best_score_:.4f} | Test={auc_xgb:.4f}')

# ── LightGBM ──────────────────────────────────────────────────────────────
if LGB_AVAILABLE:
    lgb_dist = {
        'n_estimators'    : randint(100, 500),
        'num_leaves'      : randint(20, 150),
        'learning_rate'   : loguniform(0.005, 0.3),
        'subsample'       : uniform(0.5, 0.5),
        'colsample_bytree': uniform(0.4, 0.6),
        'reg_alpha'       : loguniform(1e-4, 10),
        'reg_lambda'      : loguniform(1e-4, 10),
        'min_child_samples': randint(5, 50),
    }
    rs_lgb = RandomizedSearchCV(
        lgb.LGBMClassifier(random_state=42, verbose=-1),
        lgb_dist, n_iter=40, cv=skf, scoring='roc_auc',
        n_jobs=-1, random_state=42
    )
    rs_lgb.fit(Xc_tr_sc, yc_tr)
    auc_lgb = roc_auc_score(yc_te, rs_lgb.predict_proba(Xc_te_sc)[:,1])
    boosting_results.append({'Model':'LightGBM','CV AUC':round(rs_lgb.best_score_,4),
                              'Test AUC':round(auc_lgb,4),
                              'Best Params':str(rs_lgb.best_params_)})
    print(f'LightGBM : CV={rs_lgb.best_score_:.4f} | Test={auc_lgb:.4f}')

# ── CatBoost ──────────────────────────────────────────────────────────────
if CAT_AVAILABLE:
    cat_dist = {
        'iterations'   : randint(100, 500),
        'depth'        : randint(3, 10),
        'learning_rate': loguniform(0.005, 0.3),
        'l2_leaf_reg'  : loguniform(1, 20),
        'rsm'          : uniform(0.5, 0.5),
        'subsample'    : uniform(0.5, 0.5),
    }
    rs_cat = RandomizedSearchCV(
        CatBoostClassifier(random_seed=42, verbose=0),
        cat_dist, n_iter=30, cv=skf, scoring='roc_auc',
        n_jobs=1, random_state=42
    )
    rs_cat.fit(Xc_tr_sc, yc_tr)
    auc_cat = roc_auc_score(yc_te, rs_cat.predict_proba(Xc_te_sc)[:,1])
    boosting_results.append({'Model':'CatBoost','CV AUC':round(rs_cat.best_score_,4),
                              'Test AUC':round(auc_cat,4),
                              'Best Params':str(rs_cat.best_params_)})
    print(f'CatBoost : CV={rs_cat.best_score_:.4f} | Test={auc_cat:.4f}')

# ── GradientBoosting (sklearn, always available) ───────────────────────────
gb_dist = {
    'n_estimators' : randint(100, 400),
    'max_depth'    : randint(2, 8),
    'learning_rate': loguniform(0.005, 0.3),
    'subsample'    : uniform(0.5, 0.5),
    'max_features' : uniform(0.3, 0.7),
    'min_samples_leaf': randint(1, 15),
}
rs_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_dist, n_iter=40, cv=skf, scoring='roc_auc',
    n_jobs=-1, random_state=42
)
rs_gb.fit(Xc_tr_sc, yc_tr)
auc_gb = roc_auc_score(yc_te, rs_gb.predict_proba(Xc_te_sc)[:,1])
boosting_results.append({'Model':'GradientBoosting','CV AUC':round(rs_gb.best_score_,4),
                          'Test AUC':round(auc_gb,4),
                          'Best Params':str(rs_gb.best_params_)})
print(f'GBM      : CV={rs_gb.best_score_:.4f} | Test={auc_gb:.4f}')

if boosting_results:
    boost_df = pd.DataFrame(boosting_results).sort_values('Test AUC', ascending=False)
    print('\nBoosting Model Leaderboard:')
    print(boost_df[['Model','CV AUC','Test AUC']].to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(boost_df)); w = 0.35
    ax.bar(x - w/2, boost_df['CV AUC'],   w, label='CV AUC',
           color=COLORS['primary'], alpha=0.85)
    ax.bar(x + w/2, boost_df['Test AUC'], w, label='Test AUC',
           color=COLORS['accent'], alpha=0.85)
    for i, (cv, te) in enumerate(zip(boost_df['CV AUC'], boost_df['Test AUC'])):
        ax.text(i-w/2, cv+0.002, f'{cv:.4f}', ha='center', fontsize=10)
        ax.text(i+w/2, te+0.002, f'{te:.4f}', ha='center', fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(boost_df['Model'], fontsize=11)
    ax.set_ylabel('ROC-AUC', fontsize=11)
    ax.set_title('Boosting Models — RandomizedSearch Tuning Comparison',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.set_ylim([0.7, 1.05])
    plt.tight_layout(); plt.show()

---
## 7️⃣ Parameter Importance — Which Hyperparameters Matter Most?

> After Random Search, we can analyze **which hyperparameters correlate  
> most strongly with CV performance** — revealing which ones are worth  
> tuning carefully vs which ones don't matter much.


In [ ]:
cv_df_rf = pd.DataFrame(rs_rf.cv_results_)

# Extract numeric parameter columns
param_cols = [c for c in cv_df_rf.columns if c.startswith('param_')]
numeric_params = {}
for col in param_cols:
    try:
        vals = pd.to_numeric(cv_df_rf[col], errors='coerce')
        if vals.notna().sum() > n_iter * 0.5:
            numeric_params[col.replace('param_', '')] = vals
    except Exception:
        pass

scores = cv_df_rf['mean_test_score'].values

# Correlation of each param with score
correlations = {}
for param, vals in numeric_params.items():
    mask = vals.notna()
    if mask.sum() > 10:
        corr = np.corrcoef(vals[mask], scores[mask])[0, 1]
        correlations[param] = corr

corr_df = pd.DataFrame(list(correlations.items()),
                        columns=['Parameter', 'Correlation with CV AUC']
                       ).sort_values('Correlation with CV AUC', key=abs, ascending=False)

print('Parameter Correlation with CV ROC-AUC:')
print(corr_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(17, 5))

bar_colors = [COLORS['accent'] if v > 0 else COLORS['secondary']
              for v in corr_df['Correlation with CV AUC']]
axes[0].barh(corr_df['Parameter'], corr_df['Correlation with CV AUC'],
             color=bar_colors, alpha=0.85, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_xlabel('Pearson Correlation with CV AUC', fontsize=11)
axes[0].set_title('Parameter Importance (via Correlation)',
                  fontsize=12, fontweight='bold')
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(color=COLORS['accent'],    label='Positive effect'),
    Patch(color=COLORS['secondary'], label='Negative effect'),
], fontsize=9)

# Scatter: most important parameter vs score
if corr_df['Parameter'].iloc[0] in numeric_params:
    top_param  = corr_df['Parameter'].iloc[0]
    top_vals   = numeric_params[top_param]
    mask       = top_vals.notna()
    axes[1].scatter(top_vals[mask], scores[mask],
                    c=scores[mask], cmap='RdYlGn', s=60, alpha=0.7,
                    edgecolors='white')
    axes[1].set_xlabel(top_param, fontsize=11)
    axes[1].set_ylabel('CV ROC-AUC', fontsize=11)
    axes[1].set_title(f'Most Important Parameter: {top_param}
vs CV ROC-AUC',
                      fontsize=12, fontweight='bold')

plt.suptitle('RandomizedSearch — Parameter Importance Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8️⃣ Regression Hyperparameter Tuning

> RandomizedSearchCV works identically for regression —  
> use `scoring='r2'` or `scoring='neg_root_mean_squared_error'`.


In [ ]:
# ── RF Regressor ─────────────────────────────────────────────────────────
rfr_dist = {
    'n_estimators'     : randint(50, 400),
    'max_depth'        : randint(2, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf' : randint(1, 15),
    'max_features'     : uniform(0.1, 0.9),
}
rs_rfr = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rfr_dist, n_iter=60, cv=kf,
    scoring='r2', n_jobs=-1, random_state=42,
    return_train_score=True
)
rs_rfr.fit(Xr_tr_sc, yr_tr)
r2_rfr  = r2_score(yr_te, rs_rfr.predict(Xr_te_sc))
rmse_rfr = np.sqrt(mean_squared_error(yr_te, rs_rfr.predict(Xr_te_sc)))

print('RF Regressor:')
print(f'  Best params : {rs_rfr.best_params_}')
print(f'  CV R²       : {rs_rfr.best_score_:.4f}')
print(f'  Test R²     : {r2_rfr:.4f}')
print(f'  Test RMSE   : {rmse_rfr:.4f}')

# ── GBM Regressor ─────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor
gbr_dist = {
    'n_estimators' : randint(100, 400),
    'max_depth'    : randint(2, 8),
    'learning_rate': loguniform(0.005, 0.3),
    'subsample'    : uniform(0.5, 0.5),
    'max_features' : uniform(0.3, 0.7),
}
rs_gbr = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    gbr_dist, n_iter=50, cv=kf,
    scoring='r2', n_jobs=-1, random_state=42,
    return_train_score=True
)
rs_gbr.fit(Xr_tr_sc, yr_tr)
r2_gbr   = r2_score(yr_te, rs_gbr.predict(Xr_te_sc))
rmse_gbr = np.sqrt(mean_squared_error(yr_te, rs_gbr.predict(Xr_te_sc)))

print('\nGBM Regressor:')
print(f'  Best params : {rs_gbr.best_params_}')
print(f'  CV R²       : {rs_gbr.best_score_:.4f}')
print(f'  Test R²     : {r2_gbr:.4f}')
print(f'  Test RMSE   : {rmse_gbr:.4f}')

# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, model, r2) in zip(axes, [
    ('RF Regressor',  rs_rfr, r2_rfr),
    ('GBM Regressor', rs_gbr, r2_gbr),
]):
    y_pred = model.predict(Xr_te_sc)
    resid  = yr_te.values - y_pred
    ax.scatter(y_pred, resid, color=COLORS['primary'], alpha=0.5,
               s=30, edgecolors='white')
    ax.axhline(0, color=COLORS['secondary'], linestyle='--', linewidth=2)
    ax.set_xlabel('Predicted Values', fontsize=11)
    ax.set_ylabel('Residuals', fontsize=11)
    ax.set_title(f'{name} — Residual Plot
Test R²={r2:.4f}',
                 fontsize=12, fontweight='bold')

plt.suptitle('Regression Tuning — Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9️⃣ Pipeline + RandomizedSearchCV — Leakage-Free Tuning

> Always wrap preprocessing inside a Pipeline before passing to RandomizedSearchCV.  
> Pipeline step params use the naming convention: `stepname__paramname`


In [ ]:
pipe_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(random_state=42)),
])

pipe_param_dist = {
    'clf__n_estimators' : randint(100, 400),
    'clf__max_depth'    : randint(2, 8),
    'clf__learning_rate': loguniform(0.005, 0.3),
    'clf__subsample'    : uniform(0.5, 0.5),
    'clf__max_features' : uniform(0.3, 0.7),
}

rs_pipe = RandomizedSearchCV(
    pipe_gb,
    pipe_param_dist,
    n_iter=50,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
    verbose=0,
)

# Fit on RAW X — scaler inside pipeline handles scaling per fold
rs_pipe.fit(Xc_tr, yc_tr)
test_auc_pipe = roc_auc_score(yc_te, rs_pipe.predict_proba(Xc_te)[:,1])

print('Pipeline + RandomizedSearchCV (GradientBoosting):')
print(f'  Best params  : {rs_pipe.best_params_}')
print(f'  Best CV AUC  : {rs_pipe.best_score_:.4f}')
print(f'  Test AUC     : {test_auc_pipe:.4f}')
print(f'  Overfit gap  : {rs_pipe.best_score_ - test_auc_pipe:.4f}')

# Top configs
pipe_df = pd.DataFrame(rs_pipe.cv_results_)
top5    = pipe_df.nlargest(5, 'mean_test_score')[[
    'param_clf__n_estimators','param_clf__max_depth',
    'param_clf__learning_rate','param_clf__subsample',
    'mean_test_score','mean_train_score'
]].round(4)
top5.columns = ['n_est','depth','lr','subsample','Val AUC','Train AUC']
print('\nTop 5 Pipeline Configurations:')
print(top5.to_string(index=False))

---
## 🔟 Multi-Model Leaderboard — All Tuned Models

> Final comparison of all tuned models — showing the benefit of  
> Random Search over default hyperparameters.


In [ ]:
# Baseline (default params)
baseline = {
    'RF (default)'  : RandomForestClassifier(random_state=42),
    'GBM (default)' : GradientBoostingClassifier(random_state=42),
    'LR (default)'  : LogisticRegression(max_iter=1000, random_state=42),
    'SVM (default)' : SVC(probability=True, random_state=42),
}

leaderboard_rows = []
print('Collecting baseline scores...')
for name, model in baseline.items():
    model.fit(Xc_tr_sc, yc_tr)
    auc = roc_auc_score(yc_te, model.predict_proba(Xc_te_sc)[:,1])
    cv  = cross_val_score(model, Xc_tr_sc, yc_tr,
                           cv=skf, scoring='roc_auc').mean()
    leaderboard_rows.append({
        'Model'     : name,
        'Tuned'     : '❌',
        'CV AUC'    : round(cv, 4),
        'Test AUC'  : round(auc, 4),
    })

# Tuned models
tuned = {
    'RF (tuned)'  : (rs_rf,   Xc_te_sc),
    'GBM (tuned)' : (rs_gb,   Xc_te_sc),
    'GBM Pipeline': (rs_pipe, Xc_te),
}
for name, (search, X_te) in tuned.items():
    auc = roc_auc_score(yc_te, search.predict_proba(X_te)[:,1])
    leaderboard_rows.append({
        'Model'    : name,
        'Tuned'    : '✅',
        'CV AUC'   : round(search.best_score_, 4),
        'Test AUC' : round(auc, 4),
    })

lb_final = pd.DataFrame(leaderboard_rows).sort_values('Test AUC', ascending=False)
print('\n📊 Final Leaderboard — Default vs Tuned:')
print(lb_final.to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 6))
colors = [COLORS['accent'] if t == '✅' else COLORS['primary']
          for t in lb_final['Tuned']]
bars = ax.barh(lb_final['Model'], lb_final['Test AUC'],
               color=colors, alpha=0.85, edgecolor='white')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Random baseline')
for bar, val in zip(bars, lb_final['Test AUC']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COLORS['accent'],  label='Tuned (RandomizedSearch)'),
    Patch(color=COLORS['primary'], label='Default hyperparameters'),
], fontsize=10)
ax.set_xlabel('Test ROC-AUC', fontsize=11)
ax.set_title('Leaderboard — Default vs RandomizedSearch Tuned Models',
             fontsize=13, fontweight='bold')
ax.set_xlim([0.5, 1.05])
plt.tight_layout()
plt.show()

---
## ✅ 11. Summary & Golden Rules

| Aspect | RandomizedSearchCV | GridSearchCV |
|--------|:-----------------:|:------------:|
| Search strategy | Random sampling | Exhaustive |
| Handles distributions | ✅ Yes | ❌ Lists only |
| Sample efficiency | ✅ Better for large spaces | ❌ Poor |
| Best for | ≥ 3 hyperparameters | ≤ 3 hyperparameters |
| sklearn class | `RandomizedSearchCV` | `GridSearchCV` |

### 🔑 Golden Rules

1. **Use loguniform for rates** — learning_rate, C, alpha, gamma, reg_alpha
2. **Use uniform for fractions** — subsample, colsample_bytree, max_features
3. **Use randint for integers** — n_estimators, max_depth, num_leaves
4. **n_iter=50** is a good starting point; use 100+ for large search spaces
5. **Always use Pipeline** — scaler inside CV folds, never outside
6. **Set random_state** for reproducibility in both the search and the model
7. **return_train_score=True** — check overfit gap during search
8. **Plot convergence** — if best score plateaus early, n_iter is sufficient
9. **Analyze parameter correlations** — focus tuning effort on impactful params
10. **Follow up with Bayesian Optimization** for even more sample efficiency

---

## 🔗 Next Steps

- ➡️ `07_Hyperparameter_Tuning/gridsearchcv.ipynb` — Exhaustive grid search
- ➡️ `07_Hyperparameter_Tuning/bayesian_optimization.md` — Smarter search
- ➡️ `05_Model_Evaluation/cross_validation.ipynb` — Nested CV for unbiased evaluation
- ➡️ `08_Ensemble_Learning/` — Tune stacking and blending ensembles
